<a href="https://colab.research.google.com/github/eliza-wollinger/plaina-limadora/blob/main/whitworth.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# ======================
# DADOS DO MECANISMO
# ======================

r = 25.31
a = 130.20
b = 127.50

Z_motor = 30
Z_coroa = 68
rpm_motor = 10

i = Z_coroa / Z_motor
rpm_coroa = rpm_motor / i
omega = 2 * np.pi * rpm_coroa / 60

n_frames = 360

# Escala visual
escala = 0.03

r_vis = r * escala
a_vis = a * escala
b_vis = b * escala

A = np.array([0.0, 0.0])
C = np.array([0.0, -b_vis])
yF = a_vis

CD_visual = 8

# ======================
# EQUAÇÕES
# ======================

theta_array = np.linspace(0, 2*np.pi, n_frames, endpoint=False)
theta_deg = np.degrees(theta_array)

x_array = (a + b) * (r * np.cos(theta_array)) / (
    r * np.sin(theta_array) + b
)

v_array = -omega * (a + b) * r * (
    (r + b * np.sin(theta_array)) /
    (r * np.sin(theta_array) + b)**2
)

acc_array = omega**2 * (a + b) * r * np.cos(theta_array) * (
    2*r**2 + b*r*np.sin(theta_array) - b**2
) / (r*np.sin(theta_array) + b)**3

# ======================
# FUNÇÃO DO MECANISMO
# ======================

def mecanismo_limador(theta):
    B = A + r_vis * np.array([
        np.cos(theta),
        np.sin(theta)
    ])

    v = B - C
    u = v / np.linalg.norm(v)

    D = C + CD_visual * u

    x_real = (a + b) * (r * np.cos(theta)) / (
        r * np.sin(theta) + b
    )

    x_vis = x_real * escala

    F = np.array([x_vis, yF])

    return B, D, F, x_real

# ======================
# DASHBOARD
# ======================

fig = plt.figure(figsize=(14, 8))

gs = fig.add_gridspec(
    3,
    2,
    width_ratios=[1.3, 1],
    height_ratios=[1, 1, 1]
)

ax_mec = fig.add_subplot(gs[:, 0])
ax_x = fig.add_subplot(gs[0, 1])
ax_v = fig.add_subplot(gs[1, 1])
ax_a = fig.add_subplot(gs[2, 1])

# ======================
# MECANISMO
# ======================

ax_mec.set_title("Mecanismo Whitworth")
ax_mec.set_aspect("equal")
ax_mec.set_xlim(-5, 5)
ax_mec.set_ylim(-5, 5)
ax_mec.grid(True)

linha_AB, = ax_mec.plot([], [], linewidth=3, label="Manivela AB")
linha_CD, = ax_mec.plot([], [], linewidth=3, label="Haste CD")
linha_DF, = ax_mec.plot([], [], linewidth=2, label="Biela DF")

ponto_A, = ax_mec.plot([], [], "o", markersize=6)
ponto_B, = ax_mec.plot([], [], "o", markersize=6)
ponto_C, = ax_mec.plot([], [], "o", markersize=6)
ponto_D, = ax_mec.plot([], [], "o", markersize=6)
ponto_F, = ax_mec.plot([], [], "o", markersize=6)

circulo = plt.Circle(
    A,
    r_vis,
    fill=False,
    linewidth=2,
    alpha=0.5
)
ax_mec.add_patch(circulo)

ax_mec.plot([-5, 5], [yF, yF], linewidth=2)

slider_w = 0.6
slider_h = 0.3

slider = plt.Rectangle(
    (0, 0),
    slider_w,
    slider_h,
    fill=False,
    linewidth=2
)
ax_mec.add_patch(slider)

texto = ax_mec.text(-4.7, 4.4, "", fontsize=10)

ax_mec.legend(loc="lower left")

# ======================
# GRÁFICOS
# ======================

ax_x.plot(theta_deg, x_array)
ponto_x, = ax_x.plot([], [], "o", markersize=7)
ax_x.set_title("Posição")
ax_x.set_ylabel("x [mm]")
ax_x.set_xlim(0, 360)
ax_x.grid(True)

ax_v.plot(theta_deg, v_array)
ponto_v, = ax_v.plot([], [], "o", markersize=7)
ax_v.set_title("Velocidade")
ax_v.set_ylabel("v [mm/s]")
ax_v.set_xlim(0, 360)
ax_v.grid(True)

ax_a.plot(theta_deg, acc_array)
ponto_a, = ax_a.plot([], [], "o", markersize=7)
ax_a.set_title("Aceleração")
ax_a.set_xlabel("Ângulo θ [graus]")
ax_a.set_ylabel("a [mm/s²]")
ax_a.set_xlim(0, 360)
ax_a.grid(True)

plt.tight_layout()

# ======================
# ANIMAÇÃO
# ======================

def init():
    return (
        linha_AB, linha_CD, linha_DF,
        ponto_A, ponto_B, ponto_C, ponto_D, ponto_F,
        slider, texto,
        ponto_x, ponto_v, ponto_a
    )

def update(frame):
    theta = theta_array[frame]
    theta_plot = theta_deg[frame]

    B, D, F, x_real = mecanismo_limador(theta)

    v_real = v_array[frame]
    acc_real = acc_array[frame]

    linha_AB.set_data([A[0], B[0]], [A[1], B[1]])
    linha_CD.set_data([C[0], D[0]], [C[1], D[1]])
    linha_DF.set_data([D[0], F[0]], [D[1], F[1]])

    ponto_A.set_data([A[0]], [A[1]])
    ponto_B.set_data([B[0]], [B[1]])
    ponto_C.set_data([C[0]], [C[1]])
    ponto_D.set_data([D[0]], [D[1]])
    ponto_F.set_data([F[0]], [F[1]])

    slider.set_xy((
        F[0] - slider_w / 2,
        F[1] - slider_h / 2
    ))

    ponto_x.set_data([theta_plot], [x_array[frame]])
    ponto_v.set_data([theta_plot], [v_array[frame]])
    ponto_a.set_data([theta_plot], [acc_array[frame]])

    texto.set_text(
        f"θ = {theta_plot:.1f}°\n"
        f"x = {x_real:.2f} mm\n"
        f"v = {v_real:.2f} mm/s\n"
        f"a = {acc_real:.2f} mm/s²"
    )

    return (
        linha_AB, linha_CD, linha_DF,
        ponto_A, ponto_B, ponto_C, ponto_D, ponto_F,
        slider, texto,
        ponto_x, ponto_v, ponto_a
    )

# anim = FuncAnimation(
#     fig,
#     update,
#     frames=len(theta_array),
#     init_func=init,
#     interval=30,
#     blit=True
# )

anim = FuncAnimation(
    fig,
    update,
    frames=len(theta_array),
    init_func=init,
    interval=30,
    blit=False,
    repeat=True,
    cache_frame_data=False
)

plt.close(fig)

HTML(anim.to_html5_video())